# SLM Router v2 — Qwen3-4B Multi-task Fine-tuning (Colab A100)

**Model**: `unsloth/Qwen3-4B` (4B params, bf16)  
**Task**: Intent routing (15 classes) + contextual query rewriting  
**Hardware**: NVIDIA A100 40/80GB  
**Dataset**: multitask_train_v2.jsonl (~3,800 balanced samples)  
**Export**: GGUF Q4_K_M (~2.6GB) for laptop deployment via Ollama

In [1]:
%%capture
!pip install "unsloth[colab-new]" "trl>=0.17,<0.19"

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import torch, subprocess
gpu = subprocess.check_output(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"]).decode().strip()
print(f"GPU: {gpu}")
print(f"bf16 : {torch.cuda.is_bf16_supported()}")
print(f"Torch: {torch.__version__}")
assert torch.cuda.is_bf16_supported(), "A100 required for bf16 training"

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB, 40960 MiB
bf16 : True
Torch: 2.10.0+cu128


In [3]:
import json, re, os, unicodedata
from pathlib import Path
from collections import Counter

from unsloth import FastLanguageModel
import torch
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig

import trl; print(f"TRL version: {trl.__version__}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
TRL version: 0.18.2


In [4]:
# ══════════════════ CONFIG ══════════════════
BASE_MODEL     = "unsloth/Qwen3-4B"   # 4B params, bf16
MAX_SEQ_LENGTH = 1024
LORA_R         = 32      # 4B has enough capacity
LORA_ALPHA     = 64      # alpha/r = 2
EPOCHS         = 5
BATCH_SIZE     = 4       # per device (4B needs more VRAM)
GRAD_ACCUM     = 8       # effective batch = 32
LR             = 1e-4    # conservative for larger model
WARMUP_RATIO   = 0.05

# Paths (Google Drive)
DRIVE_DIR  = Path("/content/drive/MyDrive/hanoi-router-v2")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = str(DRIVE_DIR / "outputs")

# Data — upload multitask_train_v2.jsonl & multitask_val_v2.jsonl to this folder
DATA_DIR   = Path("/content/drive/MyDrive/chatbot-hanoi-weather")
TRAIN_FILE = DATA_DIR / "multitask_train_v2.jsonl"
VAL_FILE   = DATA_DIR / "multitask_val_v2.jsonl"

print(f"Model  : {BASE_MODEL}")
print(f"LoRA   : r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Batch  : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} effective")
print(f"Train  : {TRAIN_FILE}")
print(f"Output : {OUTPUT_DIR}")

Model  : unsloth/Qwen3-4B
LoRA   : r=32, alpha=64
Batch  : 4 x 8 = 32 effective
Train  : /content/drive/MyDrive/chatbot-hanoi-weather/multitask_train_v2.jsonl
Output : /content/drive/MyDrive/hanoi-router-v2/outputs


In [5]:
SYSTEM_PROMPT = (
    "Phân loại intent và scope cho câu hỏi thời tiết Hà Nội. Trả về JSON." + chr(10)
    + chr(10)
    + "## Intents:" + chr(10)
    + "- current_weather: thời tiết NGAY LÚC NÀY (nhiệt độ, trời nắng/mưa, chung chung)" + chr(10)
    + "- hourly_forecast: diễn biến CHI TIẾT THEO GIỜ trong ngày (không chỉ hỏi mưa)" + chr(10)
    + "- daily_forecast: dự báo NHIỀU NGÀY tới (3 ngày, tuần tới, cuối tuần)" + chr(10)
    + "- weather_overview: TỔNG QUAN, tóm tắt thời tiết hôm nay/ngày mai (không hỏi thông số cụ thể)" + chr(10)
    + "- rain_query: hỏi CÓ MƯA KHÔNG, xác suất mưa, mưa bao lâu/lúc nào tạnh" + chr(10)
    + "- temperature_query: hỏi CỤ THỂ VỀ NHIỆT ĐỘ (bao nhiêu độ, nóng/lạnh)" + chr(10)
    + "- wind_query: hỏi CỤ THỂ VỀ GIÓ (gió mạnh không, hướng gió, tốc độ gió)" + chr(10)
    + "- humidity_fog_query: hỏi về ĐỘ ẨM, SƯƠNG MÙ, sương muối" + chr(10)
    + "- historical_weather: thời tiết NGÀY/TUẦN TRƯỚC, dữ liệu QUÁ KHỨ" + chr(10)
    + "- location_comparison: SO SÁNH thời tiết giữa các quận/phường/địa điểm" + chr(10)
    + "- activity_weather: thời tiết PHÙ HỢP ĐỂ LÀM hoạt động X không (chạy bộ, picnic)" + chr(10)
    + "- expert_weather_param: thông số KỸ THUẬT chuyên sâu (áp suất, UV, điểm sương, tầm nhìn)" + chr(10)
    + "- weather_alert: CẢNH BÁO nguy hiểm: bão/áp thấp, ngập, giông/lốc, rét hại, nắng nóng cực đoan" + chr(10)
    + "- seasonal_context: SO SÁNH với hôm qua/tuần trước, xu hướng, bất thường theo MÙA" + chr(10)
    + "- smalltalk_weather: chào hỏi, ngoài phạm vi, câu hỏi không liên quan thời tiết" + chr(10)
    + chr(10)
    + "## Anti-confusion rules:" + chr(10)
    + "- bay gio/bây giờ = thời điểm hiện tại -> current_weather (KHÔNG phải wind_query)" + chr(10)
    + "- gio/gió mạnh/tốc độ gió = yếu tố gió -> wind_query" + chr(10)
    + "- bão/lũ/cảnh báo -> weather_alert (KHÔNG phải rain_query)" + chr(10)
    + chr(10)
    + "## Scopes:" + chr(10)
    + "- city: toàn Hà Nội hoặc không nói rõ địa điểm" + chr(10)
    + "- district: quận/huyện hoặc địa điểm nổi tiếng (Hồ Gươm->Hoàn Kiếm)" + chr(10)
    + "- ward: phường/xã" + chr(10)
    + chr(10)
    + "## Output:" + chr(10)
    + '{"intent":"...","scope":"...","confidence":0.9}' + chr(10)
    + "Thêm rewritten_query nếu có context hoặc câu thiếu địa điểm."
)

print(SYSTEM_PROMPT[:200], "...")
print(f"Prompt length: {len(SYSTEM_PROMPT)} chars")

Phân loại intent và scope cho câu hỏi thời tiết Hà Nội. Trả về JSON.

## Intents:
- current_weather: thời tiết NGAY LÚC NÀY (nhiệt độ, trời nắng/mưa, chung chung)
- hourly_forecast: diễn biến CHI TIẾT ...
Prompt length: 1723 chars


In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,
    load_in_4bit   = False,  # A100 has enough VRAM for full bf16
)
print(f"Model loaded: {BASE_MODEL}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

==((====))==  Unsloth 2026.3.17: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-4B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: unsloth/Qwen3-4B
Parameters: 4,022,468,096


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0.05,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.17 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable: 66,060,288 / 4,088,528,384 (1.62%)


In [8]:
# Skip get_chat_template — it has bugs with Qwen3 in newer unsloth.
# Instead, set up tokenizer manually and build ChatML strings in format_record.
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

# Verify ChatML tokens exist
test_ids = tokenizer.encode("<|im_start|>", add_special_tokens=False)
print(f"pad_token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"<|im_start|> ids: {test_ids}")
assert len(test_ids) == 1, "ChatML special tokens not found in tokenizer"

pad_token: '<|PAD_TOKEN|>' (id=151669)
<|im_start|> ids: [151644]


In [10]:
VALID_INTENTS = [
    "current_weather", "hourly_forecast", "daily_forecast", "weather_overview",
    "rain_query", "temperature_query", "wind_query", "humidity_fog_query",
    "historical_weather", "location_comparison", "activity_weather",
    "expert_weather_param", "weather_alert", "seasonal_context", "smalltalk_weather",
]
VALID_SCOPES = ["city", "district", "ward"]

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_raw = load_jsonl(TRAIN_FILE)
val_raw   = load_jsonl(VAL_FILE)

# Validate
def validate(records, name):
    valid = []
    for r in records:
        out = r["output"] if isinstance(r["output"], dict) else json.loads(r["output"])
        if out["intent"] in VALID_INTENTS and out["scope"] in VALID_SCOPES:
            valid.append(r)
    print(f"{name}: {len(valid)}/{len(records)} valid")
    return valid

train_raw = validate(train_raw, "Train")
val_raw   = validate(val_raw, "Val")

# Intent distribution
from collections import Counter
intents = Counter(r["output"]["intent"] if isinstance(r["output"], dict) else json.loads(r["output"])["intent"] for r in train_raw)
ctx_count = sum(1 for r in train_raw if r.get("context"))
rw_count  = sum(1 for r in train_raw if (r["output"] if isinstance(r["output"], dict) else json.loads(r["output"])).get("rewritten_query"))
print(f"Context records: {ctx_count} ({ctx_count/len(train_raw)*100:.1f}%)")
print(f"Rewrite records: {rw_count} ({rw_count/len(train_raw)*100:.1f}%)")
for intent, count in sorted(intents.items(), key=lambda x: -x[1]):
    print(f"  {intent:<25} {count:>5} ({count/len(train_raw)*100:.1f}%)")

Train: 3828/3828 valid
Val: 672/672 valid
Context records: 810 (21.2%)
Rewrite records: 642 (16.8%)
  current_weather             256 (6.7%)
  location_comparison         256 (6.7%)
  wind_query                  256 (6.7%)
  temperature_query           255 (6.7%)
  activity_weather            255 (6.7%)
  expert_weather_param        255 (6.7%)
  smalltalk_weather           255 (6.7%)
  seasonal_context            255 (6.7%)
  rain_query                  255 (6.7%)
  humidity_fog_query          255 (6.7%)
  weather_alert               255 (6.7%)
  hourly_forecast             255 (6.7%)
  daily_forecast              255 (6.7%)
  weather_overview            255 (6.7%)
  historical_weather          255 (6.7%)


In [11]:
IM_START = "<|im_start|>"
IM_END   = "<|im_end|>"
NL       = chr(10)   # avoid backslash-n in notebook JSON

def format_record(rec):
    user_msg = str(rec.get("input", "")).strip()
    ctx = rec.get("context")
    if ctx:
        ctx_str  = json.dumps(ctx, ensure_ascii=False, separators=(",", ":"))
        user_msg = "[CONTEXT: " + ctx_str + "]" + NL + user_msg
    out = rec.get("output", {})
    if isinstance(out, str): out = json.loads(out)
    output_dict = {
        "intent":     out["intent"],
        "scope":      out["scope"],
        "confidence": round(float(out.get("confidence", 0.9)), 2),
    }
    rw = out.get("rewritten_query")
    if rw and str(rw).strip():
        output_dict["rewritten_query"] = str(rw).strip()
    text  = IM_START + "system"    + NL + SYSTEM_PROMPT + IM_END + NL
    text += IM_START + "user"      + NL + user_msg      + IM_END + NL
    text += IM_START + "assistant" + NL + json.dumps(output_dict, ensure_ascii=False) + IM_END + NL
    return text

# Test format
sample = format_record(train_raw[0])
print(sample[:400])
print("...")
print(f"Tokens: {len(tokenizer.encode(sample))}")

# Build datasets
train_texts = [format_record(r) for r in train_raw]
val_texts   = [format_record(r) for r in val_raw]

# Token stats
lengths = [len(tokenizer.encode(t)) for t in train_texts[:200]]
print(f"Avg tokens: {sum(lengths)/len(lengths):.0f}, Max: {max(lengths)}, Over {MAX_SEQ_LENGTH}: {sum(1 for l in lengths if l > MAX_SEQ_LENGTH)}")

raw_train = Dataset.from_dict({"text": train_texts})
raw_val   = Dataset.from_dict({"text": val_texts})
print(f"Train: {len(raw_train)}, Val: {len(raw_val)}")

<|im_start|>system
Phân loại intent và scope cho câu hỏi thời tiết Hà Nội. Trả về JSON.

## Intents:
- current_weather: thời tiết NGAY LÚC NÀY (nhiệt độ, trời nắng/mưa, chung chung)
- hourly_forecast: diễn biến CHI TIẾT THEO GIỜ trong ngày (không chỉ hỏi mưa)
- daily_forecast: dự báo NHIỀU NGÀY tới (3 ngày, tuần tới, cuối tuần)
- weather_overview: TỔNG QUAN, tóm tắt thời tiết hôm nay/ngày mai (khô
...
Tokens: 617
Avg tokens: 625, Max: 669, Over 1024: 0
Train: 3828, Val: 672


In [12]:
def tokenize_fn(examples):
    enc = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
        return_tensors=None,
    )
    enc["labels"] = [ids[:] for ids in enc["input_ids"]]
    return enc

train_dataset = raw_train.map(tokenize_fn, batched=True, remove_columns=["text"])
val_dataset   = raw_val.map(tokenize_fn, batched=True, remove_columns=["text"])
print(f"Train tokenized: {len(train_dataset)} samples")
print(f"Val tokenized  : {len(val_dataset)} samples")
print(f"Sample keys: {list(train_dataset[0].keys())}")

Map:   0%|          | 0/3828 [00:00<?, ? examples/s]

Map:   0%|          | 0/672 [00:00<?, ? examples/s]

Train tokenized: 3828 samples
Val tokenized  : 672 samples
Sample keys: ['input_ids', 'attention_mask', 'labels']


In [13]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer          = tokenizer,
    model              = model,
    label_pad_token_id = -100,
    pad_to_multiple_of = 8,
)
print("DataCollatorForSeq2Seq ready")

DataCollatorForSeq2Seq ready


In [14]:
training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    warmup_ratio                = WARMUP_RATIO,
    logging_steps               = 20,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 2,
    metric_for_best_model       = "eval_loss",
    load_best_model_at_end      = True,
    bf16                        = True,
    fp16                        = False,
    optim                       = "adamw_torch_fused",
    weight_decay                = 0.01,
    max_seq_length              = MAX_SEQ_LENGTH,
    label_smoothing_factor      = 0.1,
)
print("SFTConfig ready")
steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print(f"Steps/epoch: {steps_per_epoch}  |  Total: {steps_per_epoch * EPOCHS}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


SFTConfig ready
Steps/epoch: 119  |  Total: 595


In [15]:
trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    data_collator    = data_collator,
    packing          = False,
)
print("SFTTrainer ready")

SFTTrainer ready


In [16]:
trainer_stats = trainer.train()
print(f"Train loss: {trainer_stats.training_loss:.4f}")
print(f"Runtime:    {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Samples/s:  {trainer_stats.metrics['train_samples_per_second']:.1f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,828 | Num Epochs = 5 | Total steps = 600
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Epoch,Training Loss,Validation Loss
1,12.727061,1.615628
2,12.525423,1.601369
3,12.433923,1.584927
4,12.382956,1.581054
5,12.358774,1.579609


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Train loss: 13.6336
Runtime:    2229s
Samples/s:  8.6


In [17]:
adapter_dir = DRIVE_DIR / "lora_adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Adapter saved to {adapter_dir}")

Adapter saved to /content/drive/MyDrive/hanoi-router-v2/lora_adapter


In [18]:
# Eval — use model.eval() NOT FastLanguageModel.for_inference()
# (Avoids Unsloth Qwen3 KV-cache RoPE shape mismatch bug)
model.eval()

TEST_FILE = DATA_DIR / "multitask_val_v2.jsonl"  # or multitask_test_v2.jsonl
if not TEST_FILE.exists():
    TEST_FILE = VAL_FILE
test_records = []
with open(TEST_FILE, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            test_records.append(json.loads(line))
print(f"Test records: {len(test_records)}")

NL_eval = chr(10)

correct_route  = 0
total_route    = 0
rw_correct     = 0
rw_total       = 0
rw_entity_ok   = 0
norw_correct   = 0
norw_total     = 0

for rec in test_records:
    out = rec["output"] if isinstance(rec["output"], dict) else json.loads(rec["output"])
    expected_intent = out["intent"]
    expected_scope  = out["scope"]
    has_rw = bool(out.get("rewritten_query"))

    user_msg = str(rec.get("input", "")).strip()
    ctx = rec.get("context")
    if ctx:
        ctx_str  = json.dumps(ctx, ensure_ascii=False, separators=(",",":"))
        user_msg = "[CONTEXT: " + ctx_str + "]" + NL_eval + user_msg

    prompt  = "<|im_start|>system" + NL_eval + SYSTEM_PROMPT + "<|im_end|>" + NL_eval
    prompt += "<|im_start|>user"   + NL_eval + user_msg      + "<|im_end|>" + NL_eval
    prompt += "<|im_start|>assistant" + NL_eval

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=128, temperature=0.0,
            do_sample=False, use_cache=False,
        )
    raw = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    # Strip thinking tokens if present
    if "</think>" in raw:
        raw = raw[raw.rfind("</think>") + len("</think>"):].strip()

    try:
        pred = json.loads(raw)
    except json.JSONDecodeError:
        continue

    pred_intent = pred.get("intent", "")
    pred_scope  = pred.get("scope", "")
    route_ok = (pred_intent == expected_intent and pred_scope == expected_scope)
    if route_ok:
        correct_route += 1
    total_route += 1

    if has_rw:
        rw_total += 1
        if route_ok:
            rw_correct += 1
        pred_rw = pred.get("rewritten_query", "")
        if ctx and ctx.get("location") and ctx["location"].lower() in pred_rw.lower():
            rw_entity_ok += 1
    else:
        norw_total += 1
        if route_ok:
            norw_correct += 1

print("=" * 60)
print("FULL EVAL RESULTS")
print("=" * 60)
ra = correct_route / total_route if total_route else 0
print(f"Routing accuracy    : {correct_route}/{total_route} = {ra:.1%}    target >= 92%")
if rw_total:
    print(f"Rewrite routing acc : {rw_correct}/{rw_total} = {rw_correct/rw_total:.1%}    target >= 85%")
    print(f"Entity coverage     : {rw_entity_ok}/{rw_total} = {rw_entity_ok/rw_total:.1%}    target >= 70%")
if norw_total:
    print(f"No-rewrite routing  : {norw_correct}/{norw_total} = {norw_correct/norw_total:.1%}    target >= 80%")
print("=" * 60)
verdict = "PASS" if ra >= 0.92 else "FAIL"
print("Pass routing>=92% : " + verdict)

Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test records: 672


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

FULL EVAL RESULTS
Routing accuracy    : 603/672 = 89.7%    target >= 92%
Rewrite routing acc : 104/122 = 85.2%    target >= 85%
Entity coverage     : 119/122 = 97.5%    target >= 70%
No-rewrite routing  : 499/550 = 90.7%    target >= 80%
Pass routing>=92% : FAIL


In [19]:
NL_inf = chr(10)
test_cases = [
    {"name": "Basic current",   "input": "Bay gio thoi tiet Cau Giay the nao?",
     "expected_intent": "current_weather", "context": None},
    {"name": "Daily forecast",  "input": "Cuoi tuan Ha Noi the nao?",
     "expected_intent": "daily_forecast",  "context": None},
    {"name": "Wind query",      "input": "Gio o Hoang Mai manh khong?",
     "expected_intent": "wind_query",      "context": None},
    {"name": "Weather alert",   "input": "Co bao o Ha Noi khong?",
     "expected_intent": "weather_alert",   "context": None},
    {"name": "Context rewrite", "input": "Con ngay mai?",
     "expected_intent": "daily_forecast",
     "context": {"location": "Cau Giay", "intent": "current_weather", "turn": 1}},
    {"name": "Activity",        "input": "Sang mai di chay bo duoc khong?",
     "expected_intent": "activity_weather", "context": None},
]

print("=" * 60)
print("INFERENCE TEST")
print("=" * 60)
all_pass = True
for tc in test_cases:
    user_msg = tc["input"]
    ctx = tc.get("context")
    if ctx:
        ctx_str  = json.dumps(ctx, ensure_ascii=False, separators=(",",":"))
        user_msg = "[CONTEXT: " + ctx_str + "]" + NL_inf + user_msg
    prompt  = "<|im_start|>system"    + NL_inf + SYSTEM_PROMPT + "<|im_end|>" + NL_inf
    prompt += "<|im_start|>user"      + NL_inf + user_msg      + "<|im_end|>" + NL_inf
    prompt += "<|im_start|>assistant" + NL_inf
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=128, temperature=0.0, do_sample=False, use_cache=False)
    raw = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if "</think>" in raw:
        raw = raw[raw.rfind("</think>") + len("</think>"):].strip()
    try:
        pred = json.loads(raw)
    except Exception:
        pred = {"raw": raw}
    ok = pred.get("intent") == tc["expected_intent"]
    status = "OK" if ok else "FAIL"
    if not ok:
        all_pass = False
    name = tc["name"]
    inp_text = tc["input"][:60]
    print(status.rjust(4) + " [" + name + "]")
    print("     Input:  " + inp_text)
    print("     Pred:   " + json.dumps(pred, ensure_ascii=False))
    print()
print("All passed: " + str(all_pass))

Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFERENCE TEST


Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  OK [Basic current]
     Input:  Bay gio thoi tiet Cau Giay the nao?
     Pred:   {"intent": "current_weather", "scope": "district", "confidence": 0.9}



Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  OK [Wind query]
     Input:  Gio o Hoang Mai manh khong?
     Pred:   {"intent": "wind_query", "scope": "district", "confidence": 0.9}



Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  OK [Weather alert]
     Input:  Co bao o Ha Noi khong?
     Pred:   {"intent": "weather_alert", "scope": "city", "confidence": 0.9}



Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  OK [Context rewrite]
     Input:  Con ngay mai?
     Pred:   {"intent": "daily_forecast", "scope": "district", "confidence": 0.91, "rewritten_query": "Thời tiết ngày mai ở quận Cầu Giấy thế nào?"}

  OK [Activity]
     Input:  Sang mai di chay bo duoc khong?
     Pred:   {"intent": "activity_weather", "scope": "city", "confidence": 0.9}

All passed: True


In [ ]:
# Free disk space: delete training checkpoints (adapter already saved to Drive)
import shutil, subprocess

# Show current disk usage
result = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
print(result.stdout)

# Delete checkpoints (adapter is already saved to Drive)
output_path = Path(OUTPUT_DIR)
if output_path.exists():
    for ckpt in sorted(output_path.glob("checkpoint-*")):
        print(f"Deleting {ckpt.name}...")
        shutil.rmtree(ckpt)

# Clear torch cache
torch.cuda.empty_cache()
import gc; gc.collect()

# Show freed space
result = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
print(result.stdout)


In [28]:
# Export chỉ Q4_K_M (nhỏ hơn, ~2.6GB) — bỏ Q8_0 để tiết kiệm disk
# Lưu thẳng vào Drive thay vì /content
gguf_dir = str(DRIVE_DIR / "gguf" / "q4_k_m")
print("Exporting GGUF Q4_K_M (for laptop)...")
model.save_pretrained_gguf(
    gguf_dir,
    tokenizer,
    quantization_method="q4_k_m",
)
print("Q4_K_M done!")

# Show file sizes
for f in Path(gguf_dir).iterdir():
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.1f} MB")


Exporting GGUF Q4_K_M (for laptop)...
Unsloth: Merging model weights to 16-bit format...


RuntimeError: Failed to save/merge model: Unsloth: Failed saving locally - no disk space left. Uploading can work luckily! Use .push_to_hub instead.

In [ ]:
# Fallback: push trực tiếp lên HuggingFace (không cần disk space)
from google.colab import userdata
hf_token = ""

model.push_to_hub_gguf(
    "daredevil467/qwen3-4b-hanoi-weather-router-v2",
    tokenizer,
    quantization_method="q4_k_m",
    token=hf_token,
)
print("Pushed Q4_K_M to HuggingFace!")


Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 2 files from cache to `/tmp/unsloth_gguf_h4gvmzpd`:   0%|          | 0/2 [00:00<?, ?it/s]
Unsloth: Copying 2 files from cache to `/tmp/unsloth_gguf_h4gvmzpd`:  50%|█████     | 1/2 [00:09<00:09,  9.52s/it]
Unsloth: Copying 2 files from cache to `/tmp/unsloth_gguf_h4gvmzpd`: 100%|██████████| 2/2 [00:17<00:00,  8.86s/it]


Successfully copied all 2 files from cache to `/tmp/unsloth_gguf_h4gvmzpd`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_h4gvmzpd_gguf/Qwen3-4B.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_h4gvmzpd_gguf/Qwen3-4B.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /tmp/unsloth_gguf_h4gvmzpd_gguf/Qwen3-4B.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /tmp/unsloth_gguf_h4gvmzpd_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /tmp/unsloth_gguf_h4gvmzpd_gguf/Modelfile
Unsloth: Uploading GGUF to Huggingface Hub...
Uploading Qwen3-4B.Q4_K_M.gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...gguf/Qwen3-4B.Q4_K_M.gguf:   3%|2         | 64.0MB / 2.50GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/daredevil467/qwen3-4b-hanoi-weather-router-v2
Unsloth: Cleaning up temporary files...
Pushed Q4_K_M to HuggingFace!


In [30]:
modelfile_content = (
    "FROM ./unsloth.Q4_K_M.gguf" + chr(10)
    + chr(10)
    + "TEMPLATE " + chr(34)*3 + chr(10)
    + "{{- if .System }}<|im_start|>system" + chr(10)
    + "{{ .System }}<|im_end|>" + chr(10)
    + "{{ end }}<|im_start|>user" + chr(10)
    + "{{ .Prompt }}<|im_end|>" + chr(10)
    + "<|im_start|>assistant" + chr(10)
    + "{{ .Response }}<|im_end|>" + chr(10)
    + chr(34)*3 + chr(10)
    + chr(10)
    + "PARAMETER temperature 0" + chr(10)
    + "PARAMETER num_predict 128" + chr(10)
    + "PARAMETER stop " + chr(34) + "<|im_end|>" + chr(34) + chr(10)
    + "PARAMETER stop " + chr(34) + "<|im_start|>" + chr(34) + chr(10)
)

modelfile_path = DRIVE_DIR / "gguf" / "q4_k_m" / "Modelfile"
modelfile_path.write_text(modelfile_content, encoding="utf-8")
print(f"Modelfile saved to {modelfile_path}")
print(modelfile_content)

Modelfile saved to /content/drive/MyDrive/hanoi-router-v2/gguf/q4_k_m/Modelfile
FROM ./unsloth.Q4_K_M.gguf

TEMPLATE """
{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
{{ .Response }}<|im_end|>
"""

PARAMETER temperature 0
PARAMETER num_predict 128
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"



In [31]:
# Optional: push LoRA adapter to HuggingFace Hub
# Uncomment and set your repo name

# from google.colab import userdata
# hf_token = userdata.get("HF_TOKEN")
# HF_REPO = "your-username/qwen3-4b-hanoi-weather-router-v2"
#
# model.push_to_hub(HF_REPO, token=hf_token)
# tokenizer.push_to_hub(HF_REPO, token=hf_token)
# print(f"Pushed to https://huggingface.co/{HF_REPO}")

In [32]:
print("=" * 60)
print("  TRAINING SUMMARY")
print("=" * 60)
print(f"  Model      : {BASE_MODEL}")
print(f"  LoRA       : r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  Dataset    : {len(train_dataset)} train / {len(val_dataset)} val")
print(f"  Epochs     : {EPOCHS}")
print(f"  Train loss : {trainer_stats.training_loss:.4f}")
print(f"  Output     : {DRIVE_DIR}")
print(f"  GGUF Q4_K_M: {DRIVE_DIR / 'gguf/q4_k_m'}")
print(f"  GGUF Q8_0  : {DRIVE_DIR / 'gguf/q8_0'}")
print("=" * 60)
print()
print("Next steps:")
print("  1. Download GGUF Q4_K_M from Drive")
print("  2. ollama create hanoi-weather-router-v2 -f Modelfile")
print("  3. Test: ollama run hanoi-weather-router-v2")

  TRAINING SUMMARY
  Model      : unsloth/Qwen3-4B
  LoRA       : r=32, alpha=64
  Dataset    : 3828 train / 672 val
  Epochs     : 5
  Train loss : 13.6336
  Output     : /content/drive/MyDrive/hanoi-router-v2
  GGUF Q4_K_M: /content/drive/MyDrive/hanoi-router-v2/gguf/q4_k_m
  GGUF Q8_0  : /content/drive/MyDrive/hanoi-router-v2/gguf/q8_0

Next steps:
  1. Download GGUF Q4_K_M from Drive
  2. ollama create hanoi-weather-router-v2 -f Modelfile
  3. Test: ollama run hanoi-weather-router-v2
